# PARC2026 — Dataset Ablation Manifests V1

Static Quality Analyzerの結果から、**固定eval holdout** と V0/V1/V2 のtraining episode manifestを決定論的に生成します。

Filteringとtask balancingは別軸に保ちます。公開fallbackはData Factory開発用proxyであり、Run A固定前には運営 `libero_combined_20hz` で再生成します。

## Self-contained preflight
`00` / `30` を同じruntimeで先に実行していなくても、このNotebook単体でworkspace・repo・必要なstatic metricsまで準備します。

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
print('python:', sys.version)
print('platform:', platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
ROOT = Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git', str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

## Dataset / Static metricsを準備
運営trajectoryがなければcompact public `lerobot/libero_plus` v3のmeta + parquetだけを取得します。動画は不要です。

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30','pyarrow>=16','pandas>=2'], check=True)
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
PUBLIC_DATASET = 'lerobot/libero_plus'
PUBLIC_ROOT = ROOT/'datasets'/'public_libero_plus_v3_quality'
ORGANIZER_ROOT = ROOT/'datasets'/'libero_combined_20hz'
configured = os.environ.get('PARC_DATASET_ROOT')
def ready(p): return (p/'meta'/'info.json').exists() and any(p.glob('data/**/*.parquet'))
if configured and ready(Path(configured)):
    DATASET_ROOT = Path(configured); DATASET_ID = os.environ.get('PARC_DATASET_ID','local/libero_combined_20hz'); DATASET_REVISION = None
elif ready(ORGANIZER_ROOT):
    DATASET_ROOT = ORGANIZER_ROOT; DATASET_ID = 'local/libero_combined_20hz'; DATASET_REVISION = None
else:
    api = HfApi(); info = api.dataset_info(PUBLIC_DATASET); DATASET_REVISION = info.sha
    files = api.list_repo_files(PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION)
    data_files = sorted(f for f in files if f.startswith('data/') and f.endswith('.parquet'))
    for filename in ['meta/info.json','meta/tasks.parquet',*data_files]:
        hf_hub_download(repo_id=PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION, filename=filename, local_dir=str(PUBLIC_ROOT))
    DATASET_ROOT = PUBLIC_ROOT; DATASET_ID = PUBLIC_DATASET
print('dataset:', DATASET_ID, '@', DATASET_REVISION)
print('root:', DATASET_ROOT)

In [ ]:
STATIC_OUT = ROOT/'outputs'/'static_quality_v1'
METRICS = STATIC_OUT/'episode_quality_metrics.csv'
if not METRICS.exists():
    print('static metrics not found; generating now...')
    subprocess.run([sys.executable, str(REPO/'tools/data/static_quality_analyzer.py'), '--root', str(DATASET_ROOT), '--out', str(STATIC_OUT), '--smooth-window','5','--robust-z-threshold','5.0'], check=True)
print('metrics:', METRICS)
m = pd.read_csv(METRICS)
print('episodes:', len(m), 'tasks:', m.task_index.nunique(), 'review:', int(m.quality_review_candidate.sum()))

## Manifest生成
全variantから同じ **2 episodes/taskのOK-only固定holdout** を除外します。V2はLeRobotのframe-level samplingを考慮して、task frame exposureを `sqrt` 方向へdownsampleします。

In [ ]:
MANIFEST_OUT = ROOT/'outputs'/'dataset_ablation_manifests_v1'
cmd = [sys.executable, str(REPO/'tools/data/build_dataset_ablation_manifests.py'), '--metrics-csv', str(METRICS), '--out', str(MANIFEST_OUT), '--dataset-id', DATASET_ID, '--seed','20260830','--eval-per-task','2']
if DATASET_REVISION:
    cmd += ['--dataset-revision', DATASET_REVISION]
subprocess.run(cmd, check=True)
matrix = json.loads((MANIFEST_OUT/'run_matrix.json').read_text())
print('manifest dir:', MANIFEST_OUT)

In [ ]:
rows = []
for name, s in matrix['variants'].items():
    rows.append({'variant': name, 'episodes': s['episode_count'], 'frames': s['frame_count'], 'tasks': s['task_count']})
display(pd.DataFrame(rows))
print('fixed eval:', matrix['fixed_eval']['episode_count'], 'episodes /', matrix['fixed_eval']['task_count'], 'tasks')
print('cheap order:', matrix['cheap_ablation_order'])
print('skip:', matrix['skip_reason'])

## Gate
公開LIBERO-plus proxyでは、Static Quality結果が14,347 episodesなら概ね `V0=14,267`、`V1 multi-flag=14,173`、`V1 all-review=13,895` になります（80 held-out後）。V2はframe-based sqrtなので件数は固定値そのものを目的にせず、task frame shareがRawより平坦化していることを確認します。

In [ ]:
eval_manifest = json.loads((MANIFEST_OUT/'FIXED_EVAL_HOLDOUT.json').read_text())
assert eval_manifest['summary']['task_count'] == m.task_index.nunique()
assert set(eval_manifest['episode_ids']).isdisjoint(set(json.loads((MANIFEST_OUT/'V0_RAW.json').read_text())['episode_ids']))
if len(m) == 14347 and int(m.quality_review_candidate.sum()) == 372:
    expected = {'V0_RAW':14267, 'V1_MULTI_FLAG_PRUNED_EXPERIMENTAL':14173, 'V1_ALL_REVIEW_PRUNED_EXPERIMENTAL':13895}
    for name, n in expected.items():
        assert matrix['variants'][name]['episode_count'] == n, (name, matrix['variants'][name]['episode_count'])
print('Dataset Manifest Gate: PASS')

## 次
`50_pi05_dataset_ablation.ipynb` でこのmanifestを `DatasetConfig.episodes` に渡し、π0.5を固定したcheap screeningを実行します。**training lossだけで最終variantを決めず**、shortlist後に固定評価/simulator evidenceへ進めます。